[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Shapes and Broadcasting &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


The stations' features as a tensor in each framework, as the worked examples make them.

In [2]:
table = torch.from_numpy(features)
table_tf = tf.constant(features)

**1.** Reshape `torch.arange(24)` into the shape (2, 3, 4), then into (4, -1), and print both shapes. Do the same with `tf.range` and `tf.reshape`.

In [3]:
numbers = torch.arange(24)
print(numbers.reshape(2, 3, 4).shape, numbers.reshape(4, -1).shape)

numbers_tf = tf.range(24)
print(tf.reshape(numbers_tf, (2, 3, 4)).shape, tf.reshape(numbers_tf, (4, -1)).shape)

torch.Size([2, 3, 4]) torch.Size([4, 6])
(2, 3, 4) (4, 6)


Both shapes hold the same 24 values, and `-1` became 6.

**2.** Take the station in row 10 of `features` as a batch of one, in PyTorch and in TensorFlow, and print both shapes.

In [4]:
print(table[10].unsqueeze(0).shape)
print(tf.expand_dims(table_tf[10], 0).shape)

torch.Size([1, 4])
(1, 4)


Both add a first dimension of size 1, the shape a layer expects for one station.

**3.** Find the lowest latitude among the stations and the row it is in, in PyTorch with `min(dim=0)` and in TensorFlow with `tf.reduce_min` and `tf.argmin`.

In [5]:
lowest = table[:, 0].min(dim=0)
print(round(lowest.values.item(), 2), lowest.indices.item())

latitudes_tf = table_tf[:, 0]
print(round(float(tf.reduce_min(latitudes_tf)), 2), int(tf.argmin(latitudes_tf)))

58.01 530
58.01 530


The southernmost station is at latitude 58.01, in row 530, in both frameworks.

**4.** Take the elevation column of `table_tf` with a slice, so that it keeps the shape (1200, 1), subtract the mean elevation found with `tf.reduce_mean` and `keepdims=True`, and print the result's shape and its first three values rounded to one place.

In [6]:
elevation = table_tf[:, 2:3]
above_mean = elevation - tf.reduce_mean(elevation, axis=0, keepdims=True)
print(elevation.shape, above_mean.shape)
print(above_mean[:3].numpy().round(1))

(1200, 1) (1200, 1)
[[  65.2]
 [  18.2]
 [-208.8]]


The first station is 65.2 meters above the stations' mean elevation of 292.8 meters, and the third 208.8 meters below it. The slice `2:3` kept the column two-dimensional, so the mean, of shape (1, 1), was subtracted down all 1,200 rows.

**5.** Use broadcasting on purpose, subtracting the stations' latitudes as a row of shape (1, 1200) from the same latitudes as a column of shape (1200, 1). Print the result's shape and its largest value, and check that value against the largest latitude minus the smallest.

In [7]:
latitudes = table[:, 0]
gaps = latitudes.unsqueeze(1) - latitudes.unsqueeze(0)
print(gaps.shape)
print(round(gaps.max().item(), 2), round((latitudes.max() - latitudes.min()).item(), 2))

torch.Size([1200, 1200])
12.97 12.97


Each entry is one station's latitude minus another's, so the table holds every pair, and its largest value, 12.97 degrees, is the northernmost latitude minus the southernmost.

---

&#8592; **Back to:** [Shapes and Broadcasting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/03-shapes-and-broadcasting.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
